# Aula 20 — API de leitura e camada de dados

Até agora vocês sempre leram dado de um arquivo: um CSV exportado, uma página raspada, uma tabela salva no disco. Hoje a aula tem duas metades que se juntam no fim.

Na primeira, a mão vira: você vai **consumir uma API** de verdade (um serviço que devolve dado já organizado, sem HTML no meio) e depois vai **criar a sua própria API**, um servidor que responde pedidos de outros programas, com FastAPI.

Na segunda, você vai parar de servir dado a partir de um CSV solto e passar a guardá-lo em **Parquet**, um formato de arquivo feito para dado tabular (menor, mais rápido de ler e que lembra o tipo de cada coluna). No fim, a sua API vai responder lendo do arquivo Parquet, não do CSV.

Esse é o alicerce técnico do projeto final (backend + frontend), onde o padrão é exatamente esse: uma API que lê de uma camada de dados (aqui, um arquivo Parquet). **Esta aula não cai na A1** (a prova cobre até a Aula 13); ela existe para preparar o Módulo 8.

## 1. Revisão rápida

Três coisas que já apareceram antes e voltam hoje:

- **Requisição HTTP e status code** (Aula 8): `requests.get(url)` pede um recurso pro servidor; `resposta.status_code` diz se deu certo (`200`) ou não (`404`, `500`...). Hoje o recurso pedido não é mais uma página HTML, é dado puro.
- **Dicionários e listas Python**: um dicionário guarda pares chave/valor (`{"nome": "Ana", "idade": 25}`); uma lista de dicionários é uma tabela, uma linha por dicionário. JSON (JavaScript Object Notation) é só um jeito de escrever isso como texto, pra trafegar entre programas.
- **Serialização**: transformar uma estrutura de dados (dicionário, lista) num formato que dá pra guardar ou enviar (texto, JSON, CSV) chama-se serializar. O caminho inverso (texto virando dicionário de novo) chama-se desserializar (ou "parsear"). Hoje isso acontece com um único método: `.json()`.

Se algum desses pontos ainda estiver nebuloso, vale revisitar a Aula 8 (`requests` e status code) antes de seguir.

## 2. REST: o que é

**REST** é um estilo de organizar uma API (uma forma combinada de expor dado ou funcionalidade pela internet, pra outros programas consumirem). Não é uma tecnologia específica, é uma convenção. As ideias centrais:

- **Endpoint**: um endereço (URL) que representa um recurso. Por exemplo, `/produtos` representa "a lista de produtos"; `/produtos/7` representa "o produto de id 7".
- **Método HTTP**: o verbo da ação sobre aquele recurso. Os dois que mais importam hoje:
  - **GET**: pedir um recurso, sem alterar nada no servidor (é o que fizemos a aula inteira com `requests.get()`).
  - **POST**: enviar dado novo pro servidor, geralmente para criar algo (por exemplo, cadastrar um produto). A aula de hoje foca em GET; POST aparece só de relance, pra você reconhecer o nome quando vir numa documentação.
- **Parâmetro**: informação extra que vai junto do pedido. Dois tipos comuns:
  - **Parâmetro de rota** (*path parameter*): faz parte do endereço, como o `7` em `/produtos/7`.
  - **Parâmetro de query** (*query parameter*): vem depois de `?` na URL, como `?cidade=Rio` em `/produtos?cidade=Rio`.
- **Resposta**: o que o servidor devolve, normalmente como JSON, acompanhado de um status code.

Uma API REST, então, é um conjunto de endpoints, cada um respondendo a um ou mais métodos HTTP, trocando dado em JSON. É exatamente o que vamos consumir na Seção 3 e, depois, o que vamos construir na Seção 6.

## 3. Consumindo uma API pública: ViaCEP

Vamos usar o [ViaCEP](https://viacep.com.br/), uma API pública brasileira, gratuita e sem necessidade de chave, que devolve dados de endereço a partir de um CEP. O endpoint é:

```
https://viacep.com.br/ws/{cep}/json/
```

Repare que `{cep}` é um **parâmetro de rota** (Seção 2): ele faz parte do próprio endereço, não vem depois de `?`.

A diferença para a Aula 8 é o método usado na resposta: lá, `resposta.text` trazia HTML puro, que precisava de BeautifulSoup pra virar algo útil. Aqui, a resposta já é JSON, e `.json()` transforma isso direto num dicionário Python, sem parser nenhum.

In [19]:
import requests  # já usado na Aula 8, para fazer requisições HTTP

cep = "01310930"  # CEP da Avenida Paulista, em São Paulo (só números, sem traço)
url = f"https://viacep.com.br/ws/{cep}/json/"  # {cep} entra na URL como parâmetro de rota

resposta = requests.get(url)  # faz a requisição GET
print(f"Status: {resposta.status_code}")  # sempre confira antes de seguir (hábito da Aula 8)


Status: 200


**O que observar:** o status veio `200`, então a requisição chegou e voltou certinho. Até aqui é idêntico à Aula 8. A diferença começa agora.

In [20]:
endereco = resposta.json()  # transforma o texto da resposta (JSON) direto num dicionário Python
# repare: não precisou de BeautifulSoup nem de nenhum parser manual, o próprio requests já entende JSON

print(type(endereco))  # confirma que virou um dicionário Python de verdade
endereco  # mostra o dicionário inteiro


<class 'dict'>


{'cep': '01310-930',
 'logradouro': 'Avenida Paulista',
 'complemento': '2100',
 'unidade': 'Banco Safra S.A',
 'bairro': 'Bela Vista',
 'localidade': 'São Paulo',
 'uf': 'SP',
 'estado': 'São Paulo',
 'regiao': 'Sudeste',
 'ibge': '3550308',
 'gia': '1004',
 'ddd': '11',
 'siafi': '7107'}

**O que observar:** `endereco` já é um dicionário Python normal. Dá pra acessar qualquer campo como faria com qualquer outro dicionário: `endereco["logradouro"]`, `endereco["localidade"]`, e por aí vai. Não tem HTML, não tem tag, não tem seletor. É por isso que consumir uma API costuma ser mais direto que fazer scraping: quem publicou a API já organizou o dado pra você.

In [21]:
print(f"Logradouro: {endereco['logradouro']}")
print(f"Bairro: {endereco['bairro']}")
print(f"Cidade: {endereco['localidade']} - {endereco['uf']}")


Logradouro: Avenida Paulista
Bairro: Bela Vista
Cidade: São Paulo - SP


## 4. Normalizando a resposta: de um CEP para uma lista de dicionários

Uma chamada só devolve um dicionário. O padrão útil (igual ao scraping da Aula 8) é repetir a chamada para vários CEPs e ir empilhando só os campos que interessam numa lista de dicionários, pronta para virar tabela.

In [22]:
ceps_para_consultar = ["01310930", "20040020", "70150900"]  # Paulista (SP), Cinelândia (RJ), Esplanada (DF)

enderecos_normalizados = []  # lista vazia que vai guardar um dicionário "limpo" por CEP

for cep in ceps_para_consultar:  # percorre cada CEP da lista
    url = f"https://viacep.com.br/ws/{cep}/json/"
    resposta = requests.get(url)

    if resposta.status_code == 200:  # só processa se a requisição deu certo
        dado = resposta.json()  # dicionário bruto devolvido pela API

        if "erro" not in dado:  # o ViaCEP devolve {"erro": true} quando o CEP não existe
            enderecos_normalizados.append({
                "cep": dado["cep"],
                "logradouro": dado["logradouro"],
                "bairro": dado["bairro"],
                "cidade": dado["localidade"],
                "uf": dado["uf"],
            })  # guarda só os campos que interessam, com nomes de coluna previsíveis
        else:
            print(f"CEP {cep} não encontrado.")
    else:
        print(f"Falha ao consultar {cep}: status {resposta.status_code}")

enderecos_normalizados  # lista de dicionários, pronta para virar tabela (Aulas 5 e 8 já fizeram esse passo com CSV)


[{'cep': '01310-930',
  'logradouro': 'Avenida Paulista',
  'bairro': 'Bela Vista',
  'cidade': 'São Paulo',
  'uf': 'SP'},
 {'cep': '20040-020',
  'logradouro': 'Praça Pio X',
  'bairro': 'Centro',
  'cidade': 'Rio de Janeiro',
  'uf': 'RJ'},
 {'cep': '70150-900',
  'logradouro': 'Praça dos Três Poderes',
  'bairro': 'Zona Cívico-Administrativa',
  'cidade': 'Brasília',
  'uf': 'DF'}]

**O que observar:** o dicionário bruto do ViaCEP vem com campos que talvez você nem use (`unidade`, `ibge`, `gia`, `siafi`...). Normalizar é justamente escolher só o que interessa e dar nomes de coluna consistentes, o mesmo raciocínio de quando você escolhia quais campos extrair de um HTML na Aula 8.

Repare também no `if "erro" not in dado:`. É assim que o ViaCEP sinaliza CEP inexistente: em vez de um status diferente de `200`, ele devolve `200` mesmo, só que com `{"erro": true}` no corpo. **Cada API tem sua própria convenção para erros**, por isso o próximo passo (ler a documentação) importa tanto quanto o código.

## 5. Lendo a documentação de uma API

Antes de usar qualquer API que você não conhece, vale visitar a documentação dela. É lá que você descobre:

- Quais endpoints existem e o que cada um faz.
- Quais parâmetros são obrigatórios e quais são opcionais.
- O formato exato da resposta (quais campos vêm, em que tipo).
- Se precisa de autenticação (Seção 6) e quais são os limites de uso (*rate limit*: quantas requisições por minuto/dia são permitidas).

A [documentação do ViaCEP](https://viacep.com.br/) é um bom exemplo de documentação simples: mostra o formato da URL, um exemplo de resposta em JSON, e as variações do endpoint (buscar por endereço em vez de CEP, devolver XML em vez de JSON). APIs maiores (como a do Twitter/X ou a da OpenAI) têm documentações bem mais extensas, mas a lógica de leitura é a mesma: **ache o endpoint, veja o exemplo de resposta, teste com um caso simples antes de programar em cima**.

## 6. Autenticação por chave (API key)

O ViaCEP não pede nada além da URL, mas isso é exceção: a maioria das APIs úteis (previsão do tempo, redes sociais, serviços de IA) exige uma **chave de API** (*API key*), um código que identifica quem está fazendo a requisição.

A chave costuma entrar de duas formas:

- **Como parâmetro de query:** `https://api.exemplo.com/dados?api_key=SUACHAVEAQUI`
- **Como cabeçalho (header) da requisição**, o jeito mais comum em APIs sérias:

```python
# Exemplo conceitual, não roda (API fictícia, sem chave real)
import requests

chave = "sua_chave_aqui"  # NUNCA escreva a chave direto assim num arquivo que vai pro Git
cabecalhos = {"Authorization": f"Bearer {chave}"}

resposta = requests.get("https://api.exemplo.com/dados", headers=cabecalhos)
```

**ATENÇÃO:** chave de API é senha. Duas regras que já apareceram nas Aulas 2 e 3 valem de novo, com mais força ainda:

1. **Nunca deixe a chave escrita direto no código** (*hardcoded*). Se o arquivo for parar no GitHub, a chave vaza, mesmo que o repositório seja apagado depois (histórico do Git guarda tudo).
2. **Guarde a chave num arquivo `.env`**, que fica listado no `.gitignore` (a mesma convenção da Aula 2, lembra do `exemplos/.env.example`?). O código lê a chave do `.env` em vez de tê-la escrita nele.

```python
# Exemplo conceitual de como carregar uma chave de um .env (não roda nesta aula)
import os
from dotenv import load_dotenv  # biblioteca python-dotenv, precisaria entrar no requirements.txt

load_dotenv()  # lê o arquivo .env da pasta do projeto
chave = os.getenv("API_KEY")  # pega o valor da variável API_KEY, sem nunca escrevê-la no código
```

Nesta aula não vamos usar nenhuma chave real (o ViaCEP e o servidor que vamos criar não precisam), mas o padrão acima é o que você vai usar sempre que uma API pedir autenticação daqui pra frente.

## 7. Criando sua própria API com FastAPI

Até aqui, você foi **cliente** de APIs (quem pede). Agora você vira **servidor** (quem responde). [FastAPI](https://fastapi.tiangolo.com/) é um framework Python para construir APIs: você escreve funções Python normais, decora com `@app.get(...)`, e ele cuida de toda a parte de rede, JSON e validação.

**Instalação** (na pasta da aula, com o ambiente virtual já ativo):

Windows (Prompt de Comando ou Terminal integrado do VS Code):
```cmd
uv venv .venv
uv pip install -r requirements.txt
```

Mac (Terminal ou Terminal integrado do VS Code):
```bash
uv venv .venv
uv pip install -r requirements.txt
```

O `requirements.txt` desta aula tem `fastapi` e `uvicorn` (o servidor que roda o app), `requests`, e ainda `pandas` e `pyarrow` para a parte de Parquet na segunda metade.

### O app mínimo

O código do servidor desta aula mora em `exemplos/api.py`, um arquivo separado (não uma célula deste notebook: explico o motivo já já). Ele começa assim:

```python
from fastapi import FastAPI  # classe principal do framework, cria o app

app = FastAPI(
    title="API da Aula 20",
    description="API de exemplo: catálogo pequeno de produtos, fixo no código.",
)  # cria a aplicação
```

`app` é o objeto que representa sua API inteira. Cada endpoint que você adiciona é uma função decorada, ligada a esse `app`.

**IMPORTANTE, sobre por que este código não roda como uma célula normal:** um servidor fica ligado, esperando pedidos, até alguém mandar parar. Se você tentasse "rodar" um servidor numa célula de notebook do jeito ingênuo, a célula ficaria travada para sempre (o servidor nunca "termina" sozinho), e você precisaria reiniciar o kernel. Por isso, servidor sempre roda **num terminal separado**, fora do fluxo de células. É assim que você vai rodar o seu na prática, daqui a pouco.

### A "base de dados" fixa

Antes de ler dado de um arquivo de verdade (que entra na segunda metade desta aula, a partir da Seção 10), esta primeira API guarda os dados numa lista de dicionários, direto no arquivo Python:

```python
produtos = [
    {"id": 1, "nome": "Caderno", "preco": 12.5, "categoria": "papelaria"},
    {"id": 2, "nome": "Caneta", "preco": 3.0, "categoria": "papelaria"},
    {"id": 3, "nome": "Mochila", "preco": 89.9, "categoria": "acessorios"},
]
```

Isso significa que, se o servidor for reiniciado, os dados voltam ao estado original: não tem gravação permanente ainda. A Seção 11 troca essa lista por dados lidos de um arquivo Parquet.

### Endpoint de leitura simples

```python
@app.get("/produtos")  # @app.get(...) registra esta função para responder GET nesse endereço
def listar_produtos():
    return produtos  # FastAPI converte a lista de dicionários em JSON sozinho
```

`@app.get("/produtos")` é um **decorador**: ele diz ao FastAPI "quando chegar um GET em `/produtos`, chame a função `listar_produtos`". Você não precisa serializar nada manualmente (nada de `json.dumps`), o FastAPI faz isso ao ver que a função devolveu uma lista de dicionários.

### Endpoint com parâmetro de rota e validação automática

```python
@app.get("/produtos/{produto_id}")  # {produto_id} na rota vira parâmetro de rota (Seção 2)
def obter_produto(produto_id: int):  # o ": int" pede ao FastAPI para validar o tipo
    for produto in produtos:
        if produto["id"] == produto_id:
            return produto  # achou: devolve o dicionário do produto

    return {"erro": f"Nenhum produto com id {produto_id}."}  # não achou: resposta simples
```

Aqui está o pulo do gato do FastAPI: `produto_id: int` não é só documentação, é **validação de verdade**. Se alguém pedir `/produtos/abc` (uma letra, não um número), o FastAPI recusa a requisição sozinho, **antes mesmo da função rodar**, e devolve um status `422` explicando o que esperava. Vamos ver isso ao vivo na próxima seção.

## 8. Rodando o servidor e testando pela documentação automática

Chegou a hora de rodar de verdade. Abra um terminal (o integrado do VS Code funciona bem, e evita alternar de janela) **na pasta desta aula** (`aulas/20-api-e-persistencia/`) e rode:

Windows (Prompt de Comando ou Terminal integrado do VS Code):
```cmd
uv run uvicorn exemplos.api:app --reload
```

Mac (Terminal ou Terminal integrado do VS Code):
```bash
uv run uvicorn exemplos.api:app --reload
```

- `exemplos.api:app` diz ao uvicorn: "dentro da pasta `exemplos`, arquivo `api.py`, use o objeto chamado `app`".
- `--reload` faz o servidor reiniciar sozinho sempre que você salvar uma alteração no `api.py`. **Sem essa flag, o servidor continua rodando a versão antiga do código até você pará-lo e chamar de novo.**

O terminal vai mostrar algo como `Uvicorn running on http://127.0.0.1:8000`. Isso significa que o servidor está de pé, esperando pedidos. **O terminal fica ocupado enquanto o servidor roda: é esperado, é assim que um servidor funciona.** Para parar, volte nesse terminal e aperte `Ctrl+C`.

Com o servidor rodando, abra no navegador:

```
http://127.0.0.1:8000/docs
```

Essa página é o **Swagger UI**, gerado automaticamente pelo FastAPI a partir do seu código (nenhuma linha a mais foi escrita para criar essa documentação). Nela você vê cada endpoint, pode expandir, clicar em "Try it out", preencher parâmetros e mandar a requisição direto pelo navegador, sem escrever nenhum código.

**Missão:** com o servidor rodando, na página `/docs`:
1. Teste `GET /produtos` e confira que devolve os três produtos.
2. Teste `GET /produtos/{produto_id}` com `produto_id = 2` e confira que devolve só a Caneta.
3. Teste de novo com `produto_id = abc` (um texto, não um número) e observe o status `422` e a mensagem de erro que o FastAPI devolve sozinho, sem você ter escrito nenhuma validação manual.

## 9. Vendo isso funcionar aqui no notebook

Se você já tem o servidor rodando num terminal separado (Seção 8, em `http://127.0.0.1:8000`), pode testar exatamente como faria com qualquer outra API: `requests.get("http://127.0.0.1:8000/produtos")`. **ATENÇÃO:** isso só funciona com o servidor de pé em outro terminal; sem ele, dá `ConnectionError` (veja a Seção 10, "Quando der errado").

Para este notebook não depender de você já ter aberto um terminal antes de chegar aqui, as células abaixo fazem a mesma coisa de um jeito autocontido: sobem o **mesmo** `exemplos/api.py` como um processo em segundo plano, fazem algumas requisições contra ele, e depois derrubam o processo. Isso usa o módulo `subprocess`, que ainda não é conteúdo desta aula (está aqui só como demonstração, para você ver o resultado sem sair do notebook). **No seu dia a dia, o fluxo normal continua sendo o da Seção 8: servidor num terminal, à parte.**

In [23]:
import subprocess  # permite rodar outro programa (aqui, o próprio uvicorn) como um processo separado
import sys
import time

# Inicia "uv run uvicorn exemplos.api:app" em segundo plano, numa porta própria para não colidir
# com um servidor que você já tenha aberto manualmente na Seção 8 (porta 8000)
processo_servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "exemplos.api:app", "--port", "8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(2)  # dá um tempo para o servidor terminar de subir antes de bater nele

if processo_servidor.poll() is not None:  # poll() só deixa de ser None se o processo já morreu
    # se chegou até aqui, algo impediu o servidor de subir (porta ocupada, dependência faltando...)
    saida_erro = processo_servidor.stderr.read().decode(errors="replace")
    print("O servidor de demonstração não subiu. Erro capturado:")
    print(saida_erro[-1500:])  # últimas linhas costumam ter a causa (veja a Seção 10)
else:
    print("Servidor de demonstração no ar em http://127.0.0.1:8001 (processo em segundo plano).")


Servidor de demonstração no ar em http://127.0.0.1:8001 (processo em segundo plano).


In [24]:
resposta_lista = requests.get("http://127.0.0.1:8001/produtos")
print(f"GET /produtos -> status {resposta_lista.status_code}")
print(resposta_lista.json())
print()

resposta_item = requests.get("http://127.0.0.1:8001/produtos/2")
print(f"GET /produtos/2 -> status {resposta_item.status_code}")
print(resposta_item.json())
print()

resposta_invalida = requests.get("http://127.0.0.1:8001/produtos/abc")
print(f"GET /produtos/abc -> status {resposta_invalida.status_code}")  # 422: produto_id não é um int válido
print(resposta_invalida.json())  # o FastAPI explica sozinho qual campo falhou e por quê


GET /produtos -> status 200
[{'id': 1, 'nome': 'Caderno', 'preco': 12.5, 'categoria': 'papelaria'}, {'id': 2, 'nome': 'Caneta', 'preco': 3.0, 'categoria': 'papelaria'}, {'id': 3, 'nome': 'Mochila', 'preco': 89.9, 'categoria': 'acessorios'}]

GET /produtos/2 -> status 200
{'id': 2, 'nome': 'Caneta', 'preco': 3.0, 'categoria': 'papelaria'}

GET /produtos/abc -> status 422
{'detail': [{'type': 'int_parsing', 'loc': ['path', 'produto_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


In [25]:
processo_servidor.terminate()  # pede para o processo do servidor encerrar
processo_servidor.wait(timeout=5)  # espera confirmar que encerrou de fato
print("Servidor de demonstração encerrado.")


Servidor de demonstração encerrado.


**O que observar:** os dois últimos testes da célula anterior mostram a validação automática do FastAPI ao vivo: `/produtos/2` (um inteiro válido) devolve `200` com o produto certo; `/produtos/abc` (não é um inteiro) devolve `422`, sem que a função `obter_produto` tenha chegado a rodar. Esse é o comportamento que você também vai ver testando pela página `/docs`.

## 10. Guardando a coleta em Parquet

A API da primeira metade guarda tudo numa lista fixa no código. Isso não escala: some quando o servidor reinicia, e não dá para ter mais dado do que cabe escrito num `.py`.

O passo seguinte é tirar o dado do código e guardá-lo num arquivo de dados. Vocês já conhecem o CSV. Hoje vamos usar o **Parquet**, um formato de arquivo para dados tabulares que guarda **por coluna** (e não por linha, como o CSV) e comprime o que guarda.

**Por que o Parquet é um bom formato para guardar uma coleta:**

- **É menor.** Como cada coluna fica junto, valores parecidos (o mesmo autor repetido em dezenas de posts, por exemplo) comprimem muito bem. Você mede isso na célula abaixo.
- **Lembra o tipo de cada coluna.** O CSV é só texto: uma data volta como texto e você precisa converter de novo (como na Aula 10). O Parquet guarda inteiro, decimal, texto e data como tais, e devolve igual.
- **Lê só as colunas de que precisa.** Como cada coluna fica separada no arquivo, dá para pedir só duas e o resto nem é lido. Numa base grande, isso é a diferença entre esperar segundos ou minutos. É exatamente o que a API vai fazer.
- **É padrão de mercado.** pandas, R, Spark, Google BigQuery e a maioria das ferramentas de dados leem e escrevem Parquet, então o arquivo não te prende a uma ferramenta.
- **É um arquivo só, sem servidor.** Dá para copiar, mandar, guardar no Drive. Não precisa instalar nem ligar nada para usá-lo.

**O que o Parquet não é:** ele é binário, então não abre no Bloco de Notas nem no Excel (você abre com o pandas). E não serve para editar uma linha por vez: para mudar algo, você regrava o arquivo inteiro. Por isso o CSV continua ótimo para dado pequeno que gente vai abrir e olhar, e o Parquet para dado que programa vai ler.

Vamos pegar a exportação de rede social (a mesma das Aulas 11 a 13), guardar uma versão enxuta em Parquet e comparar com o CSV.

In [26]:
import pandas as pd
from pathlib import Path

posts = pd.read_csv("data/exportacao.csv", sep=";").drop_duplicates()

# guardamos só as colunas que interessam (o CSV tem 36)
colunas = ["id", "author", "author_followers", "timestamp",
           "likes", "comments", "shares", "plays", "hashtags"]
posts_slim = posts[colunas].copy()
posts_slim["timestamp"] = pd.to_datetime(posts_slim["timestamp"])  # texto vira data de verdade (Aula 10)

Path("data").mkdir(exist_ok=True)
posts_slim.to_parquet("data/coleta.parquet", index=False)  # grava o Parquet (usa a biblioteca pyarrow)
posts_slim.to_csv("data/coleta_slim.csv", index=False)     # o mesmo dado em CSV, só para comparar

tam_original = Path("data/exportacao.csv").stat().st_size
tam_csv = Path("data/coleta_slim.csv").stat().st_size
tam_parquet = Path("data/coleta.parquet").stat().st_size
print(f"CSV original (36 colunas):   {tam_original:,} bytes")
print(f"Mesmas 9 colunas em CSV:     {tam_csv:,} bytes")
print(f"Mesmas 9 colunas em Parquet: {tam_parquet:,} bytes ({tam_parquet / tam_csv:.0%} do CSV)")

CSV original (36 colunas):   1,516,646 bytes
Mesmas 9 colunas em CSV:     71,208 bytes
Mesmas 9 colunas em Parquet: 37,719 bytes (53% do CSV)


**O que observar:** o Parquet guarda os mesmos dados em pouco mais da metade do espaço do CSV equivalente (e uma fração mínima do CSV original, que tem 36 colunas). Para uma coleta pequena isso é só conveniência; quando a base tem milhões de linhas, pesa no disco, na transferência e no tempo de leitura.

Agora a segunda vantagem, o tipo das colunas, e a terceira, ler só algumas colunas:

In [27]:
de_csv = pd.read_csv("data/coleta_slim.csv")
de_parquet = pd.read_parquet("data/coleta.parquet")

# o CSV devolve a data como texto; o Parquet lembra que era data
print("timestamp lido do CSV:    ", de_csv["timestamp"].dtype)
print("timestamp lido do Parquet:", de_parquet["timestamp"].dtype)

# lendo só duas colunas: o Parquet não precisa tocar nas outras sete
so_duas = pd.read_parquet("data/coleta.parquet", columns=["author", "plays"])
so_duas.head()

timestamp lido do CSV:     str
timestamp lido do Parquet: datetime64[us]


,author,plays
0,g1,229
1,g1,10600
2,g1,38000
3,g1,13000
4,g1,61100


**O que observar:** o `timestamp` do CSV volta como texto (`str` ou `object`, dependendo da versão do pandas), e o do Parquet volta como `datetime64`, sem você converter de novo. E o `columns=[...]` do `read_parquet` devolve só as colunas pedidas. Guarde esse detalhe: é assim que a API da próxima seção lê o arquivo.

## 11. A API lendo do Parquet

Agora fecha o ciclo: em vez de servir uma lista fixa no código, a API passa a **ler o arquivo Parquet** que a Seção 10 criou. O código está em `exemplos/api_dados.py`, um segundo servidor de exemplo. As diferenças em relação ao `exemplos/api.py`:

- não há mais lista `produtos` no código; cada endpoint lê só as colunas de que precisa de `dados/coleta.parquet` (o `columns=[...]` que você acabou de ver) e devolve o resultado;
- `GET /posts` devolve os posts (com um limite, para não despejar a coleta inteira);
- `GET /autores` devolve a contagem por autor, calculada com `groupby()` (o mesmo da Aula 5) sobre o que foi lido do arquivo.

Rode num terminal separado, **na pasta `aulas/20-api-e-persistencia/`**:

Windows (Prompt de Comando ou Terminal integrado do VS Code):
```cmd
uv run uvicorn exemplos.api_dados:app --reload
```

Mac (Terminal ou Terminal integrado do VS Code):
```bash
uv run uvicorn exemplos.api_dados:app --reload
```

Depois abra `http://127.0.0.1:8000/docs` e teste os dois endpoints.

**Missão:** com o `exemplos/api_dados.py` rodando, na página `/docs`:
1. Teste `GET /autores` e confira que o resultado bate com um `posts_slim.groupby("author")` feito aqui no notebook (é a mesma conta, agora servida pela API).
2. Teste `GET /posts` com `limite = 3` e confira que voltam só 3 posts.
3. Pare o servidor com `Ctrl+C`, apague o arquivo `dados/coleta.parquet`, suba o servidor de novo e tente `GET /autores`: o erro `500` (com `FileNotFoundError` no terminal do servidor) mostra que agora a API depende do arquivo existir, não mais de uma lista embutida.

## 12. Quando der errado

- **`requests.exceptions.ConnectionError` ao consultar uma API externa (como o ViaCEP):** o `requests` não conseguiu nem chegar no servidor. Confira sua conexão de internet e se a URL está digitada certo.
- **`requests.exceptions.JSONDecodeError` (ou erro parecido) ao chamar `.json()`:** a resposta não era JSON de verdade (às vezes é uma página de erro em HTML, por exemplo). Antes de chamar `.json()`, confira `resposta.status_code` e, se precisar debugar, dê uma olhada em `resposta.text` para ver o que realmente voltou.
- **`ConnectionError` na Seção 9 (célula que depende do terminal):** o servidor não está rodando, ou está rodando numa porta diferente de `8000`. Volte ao terminal da Seção 8 e confira se a mensagem `Uvicorn running on http://127.0.0.1:8000` apareceu, sem erro embaixo.
- **Servidor não sobe, com erro de porta ocupada (`address already in use`):** já tem outro processo usando a porta `8000` (às vezes um servidor anterior que você esqueceu de parar com `Ctrl+C`). Feche o processo anterior, ou rode com outra porta: `uv run uvicorn exemplos.api:app --reload --port 8002`.
- **`ModuleNotFoundError: No module named 'uvicorn'` (ou `'fastapi'`):** as dependências não foram instaladas nesse ambiente. Rode `uv pip install -r requirements.txt` de novo, na pasta da aula, com o `.venv` certo.
- **A célula da Seção 9 ("Demonstração autocontida") não sobe o servidor, mesmo com tudo instalado:** ela usa o mesmo Python do kernel selecionado no notebook. Se o kernel do Jupyter/VS Code não for o `.venv` desta aula (por exemplo, ficou selecionado um Python global), o `uvicorn` instalado no `.venv` não é enxergado. Confira, no canto superior direito do notebook (VS Code) ou no menu Kernel (Jupyter), se o interpretador escolhido é o de `aulas/20-api-e-persistencia/.venv`, e troque se não for.
- **Status `422 Unprocessable Entity`:** não é um bug, é a validação do FastAPI funcionando. Leia o campo `"detail"` da resposta: ele diz exatamente qual parâmetro falhou (`"loc"`) e por quê (`"msg"`), por exemplo "Input should be a valid integer".
- **Alterei o `api.py` e nada mudou no navegador:** confira se o servidor foi iniciado com `--reload`. Sem essa flag, é preciso parar (`Ctrl+C`) e rodar de novo manualmente a cada alteração.
- **`ImportError: Unable to find a usable engine` ao usar `to_parquet`/`read_parquet`:** a biblioteca `pyarrow` não está instalada nesse ambiente. Rode `uv pip install -r requirements.txt` de novo, na pasta da aula, com o `.venv` certo.
- **`FileNotFoundError` em `dados/coleta.parquet` (no notebook ou no terminal do servidor):** o arquivo ainda não foi criado, ou você está rodando de outra pasta. Rode primeiro a célula da Seção 10 que faz `to_parquet(...)`, e suba o servidor na pasta da aula.
- **A API devolve `500 Internal Server Error`:** não é problema de quem chama, é erro dentro do servidor. Olhe o terminal onde o `uvicorn` está rodando: o traceback aparece lá, com a linha que falhou.
- **O `.parquet` apareceu no `git status`:** ele é gerado, não versionado. Confira se `dados/*.parquet` está no `.gitignore` desta pasta.

## Prática: faça agora

Abra `exercicios/exercicio-20-api-e-persistencia.ipynb`. Ele tem três partes: consumir uma API pública à sua escolha (parecido com a Seção 3), guardar uma coleta sua de uma aula anterior em Parquet, e completar `api_exercicio.py` para que o seu endpoint responda **lendo do Parquet**, testado pela documentação automática em `/docs`.